# Study B - Stimuli Verification (First Stage: Pre-Check)

Notebook for pre-processing data from Study A for Study B

## Setup

In [ ]:
# Setup
!pip -q install pandas numpy

import os
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


## Config

In [ ]:
# CONFIG: set Drive folder here
BASE_DIR = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports"
os.makedirs(BASE_DIR, exist_ok=True)

PATH_STIMULI = os.path.join(BASE_DIR, "studyA_dialogs_from_goodbad.csv")                # stimuli CSV
PATH_HUMAN_GOLD = os.path.join(BASE_DIR, "human_PRIMARY_gold_dialog_level.csv")        # human gold (PRIMARY)
PATH_LLM_PRIMARY = os.path.join(BASE_DIR, "llm_PRIMARY_median_subset.csv")             # llm medians for PRIMARY dialogs

# outputs
OUT_PAIR_TABLE = os.path.join(BASE_DIR, "studyB_recipe_pairs_with_deltas.csv")
OUT_PASS_POOL  = os.path.join(BASE_DIR, "studyB_pass_pool_pairs.csv")
OUT_TOP_POOL   = os.path.join(BASE_DIR, "studyB_top_pool_pairs.csv")

print("BASE_DIR:", BASE_DIR)

BASE_DIR: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports


## Load Data

In [ ]:
def _read_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)
    return df

stim = _read_csv(PATH_STIMULI)
hgold = _read_csv(PATH_HUMAN_GOLD)
llm = _read_csv(PATH_LLM_PRIMARY)

# normalize id col
for df in [stim, hgold, llm]:
    if "dialog_id" not in df.columns:
        raise ValueError(f"dialog_id missing in df columns: {list(df.columns)}")
    df["dialog_id"] = df["dialog_id"].astype(str).str.strip()

# quick peek
print("stim columns:", list(stim.columns))
print("hgold columns:", [c for c in hgold.columns if c.startswith("human_gold_")][:10], "...")
print("llm columns:", [c for c in llm.columns if c.startswith("llm_median_")][:10], "...")
print("\nRows:", {"stim": len(stim), "hgold": len(hgold), "llm": len(llm)})

# create a slim meta frame from stimuli
meta_cols = [c for c in ["dialog_id", "recipe_title", "recipe_type", "condition", "is_gold", "is_imc"] if c in stim.columns]
stim_meta = stim[meta_cols].drop_duplicates("dialog_id").copy()

# merge metadata into hgold and llm (if needed)
if "recipe_title" not in hgold.columns or "condition" not in hgold.columns:
    hgold = hgold.merge(stim_meta[["dialog_id", "recipe_title", "recipe_type", "condition"]], on="dialog_id", how="left")

if "recipe_title" not in llm.columns or "condition" not in llm.columns:
    llm = llm.merge(stim_meta[["dialog_id", "recipe_title", "recipe_type", "condition"]], on="dialog_id", how="left")

# sanity checks
assert hgold["recipe_title"].notna().all(), "Some human_gold rows are missing recipe_title after merge."
assert hgold["condition"].notna().all(), "Some human_gold rows are missing condition after merge."
assert llm["recipe_title"].notna().all(), "Some llm rows are missing recipe_title after merge."
assert llm["condition"].notna().all(), "Some llm rows are missing condition after merge."

print("merged recipe_title/condition where needed")

stim columns: ['dialog_id', 'recipe_title', 'recipe_type', 'condition', 'dialog_text', 'is_gold', 'gold_expected_min', 'gold_expected_max', 'is_imc', 'imc_expected_overall']
hgold columns: ['human_gold_truthfulness', 'human_gold_relevance', 'human_gold_clarity', 'human_gold_logic_coherence', 'human_gold_feedback_depth', 'human_gold_respect_appreciation'] ...
llm columns: ['llm_median_clarity', 'llm_median_relevance', 'llm_median_truthfulness', 'llm_median_logic_coherence', 'llm_median_respect_appreciation', 'llm_median_relational_appropriateness', 'llm_median_feedback_depth', 'llm_median_overall_quality'] ...

Rows: {'stim': 80, 'hgold': 50, 'llm': 50}
merged recipe_title/condition where needed


## Define stimulus pool for Study B
Only dialogues with Human-Gold (PRIMARY) + only recipes with good+bad

In [ ]:
# Normalize HUMAN GOLD column names to analysis schema (once)
# (so later code can use human_gold_* consistently)

hgold = hgold.copy()

# Create expected analysis columns in case the file uses different names
if "overall" in hgold.columns and "human_gold_overall_quality" not in hgold.columns:
    hgold["human_gold_overall_quality"] = pd.to_numeric(hgold["overall"], errors="coerce")

if "relation_appropriateness" in hgold.columns and "human_gold_relational_appropriateness" not in hgold.columns:
    hgold["human_gold_relational_appropriateness"] = pd.to_numeric(hgold["relation_appropriateness"], errors="coerce")

# Ensure numeric type for all human_gold_* columns
for c in [c for c in hgold.columns if c.startswith("human_gold_")]:
    hgold[c] = pd.to_numeric(hgold[c], errors="coerce")

print("OK: human gold normalized. Available human_gold_* cols:")
print([c for c in hgold.columns if c.startswith("human_gold_")])

OK: human gold normalized. Available human_gold_* cols:
['human_gold_truthfulness', 'human_gold_relevance', 'human_gold_clarity', 'human_gold_logic_coherence', 'human_gold_feedback_depth', 'human_gold_respect_appreciation', 'human_gold_overall_quality', 'human_gold_relational_appropriateness']


In [ ]:
# Start with dialog set that has human gold (PRIMARY)
pool = hgold.copy()

# Keep only regular dialogs if those flags exist (safety)
for col in ["is_gold", "is_imc"]:
    if col in pool.columns:
        pool = pool[(pd.to_numeric(pool[col], errors="coerce").fillna(0).astype(int) == 0)]

# Keep only good/bad conditions
pool["condition"] = pool["condition"].astype(str).str.lower().str.strip()
pool = pool[pool["condition"].isin(["good", "bad"])].copy()

print("Human-gold dialog pool size:", pool["dialog_id"].nunique())
print(pool["condition"].value_counts())

# group by recipe (default key: recipe_title)
RECIPE_KEY = "recipe_title"

# Determine which recipes have BOTH conditions
cond_by_recipe = (
    pool.groupby(RECIPE_KEY)["condition"]
    .apply(lambda s: set(s.tolist()))
    .reset_index(name="cond_set")
)

both = cond_by_recipe[cond_by_recipe["cond_set"].apply(lambda s: ("good" in s) and ("bad" in s))].copy()
print("Recipes with good+bad in human-gold:", len(both))

pool_both = pool.merge(both[[RECIPE_KEY]], on=RECIPE_KEY, how="inner")
print("Dialogs in recipes that have both conditions:", pool_both["dialog_id"].nunique())

Human-gold dialog pool size: 50
condition
bad     26
good    24
Name: count, dtype: int64
Recipes with good+bad in human-gold: 18
Dialogs in recipes that have both conditions: 36


## Manipulation strength per recipe pair (human & LLM)
Build pair table (good dialogue + bad dialogue per recipe)

In [ ]:
# Helper: pick exactly 1 dialog per recipe x condition
# If there are multiple per condition (rare), choose the one with highest human overall in GOOD
# and lowest human overall in BAD, to maximize separation
H_OVERALL = "human_gold_overall_quality"

if H_OVERALL not in pool_both.columns:
    raise ValueError(f"Missing {H_OVERALL} in human gold file. Available: {[c for c in pool_both.columns if c.startswith('human_gold_')]}")

def pick_one_per_cond(df_recipe: pd.DataFrame, cond: str) -> pd.DataFrame:
    d = df_recipe[df_recipe["condition"] == cond].copy()
    if d.empty:
        return d
    if cond == "good":
        return d.sort_values(H_OVERALL, ascending=False).head(1)
    else:  # bad
        return d.sort_values(H_OVERALL, ascending=True).head(1)

pairs = []
for recipe, g in pool_both.groupby(RECIPE_KEY):
    good_row = pick_one_per_cond(g, "good")
    bad_row  = pick_one_per_cond(g, "bad")
    if good_row.empty or bad_row.empty:
        continue
    pairs.append({
        RECIPE_KEY: recipe,
        "recipe_type": good_row["recipe_type"].iloc[0] if "recipe_type" in good_row.columns else "",
        "dialog_id_good": good_row["dialog_id"].iloc[0],
        "dialog_id_bad":  bad_row["dialog_id"].iloc[0],
        "human_good_overall": float(good_row[H_OVERALL].iloc[0]),
        "human_bad_overall":  float(bad_row[H_OVERALL].iloc[0]),
    })

pairs_df = pd.DataFrame(pairs)
print("Recipe-pairs built:", len(pairs_df))
pairs_df.head()

Recipe-pairs built: 18


,recipe_title,recipe_type,dialog_id_good,dialog_id_bad,human_good_overall,human_bad_overall
0,Apple Pie Filling,sweet,D040,D077,7.0,2.5
1,Au Gratin-Pumpkin Layered Casserole,savory,D034,D064,4.5,1.5
2,Bourbon Wieners,sweet,D002,D042,6.5,1.0
3,Carrot Casserole,savory,D019,D060,6.0,1.5
4,Carrot and Sweet Potato Tzimmes,savory,D037,D071,6.5,2.0


## Merge and calculate human deltas + LLM deltas

In [ ]:
# Build lookup tables
h_lookup = hgold.set_index("dialog_id")
l_lookup = llm.set_index("dialog_id")

LLM_OVERALL = "llm_median_overall_quality"
if LLM_OVERALL not in llm.columns:
    raise ValueError(f"Missing {LLM_OVERALL} in LLM file. Available: {[c for c in llm.columns if c.startswith('llm_median_')]}")

def safe_get(df_indexed, did, col):
    if did not in df_indexed.index:
        return np.nan
    return df_indexed.loc[did, col]

# Add deltas for overall + optional subscales (Positivquote (Gate) without clarity)
subscales_pos = [
    "relevance", "truthfulness", "logic_coherence",
    "respect_appreciation", "relational_appropriateness", "feedback_depth"
]

# clarity as addictional info (report only)
subscales_report = subscales_pos + ["clarity"]

for k in subscales_pos:
    hk = f"human_gold_{k}"
    lk = f"llm_median_{k}"
    if hk not in hgold.columns:
        print("WARN missing human:", hk)
    if lk not in llm.columns:
        print("WARN missing llm:", lk)

# Fill overall deltas + LLM values
rows = []
for _, r in pairs_df.iterrows():
    did_g = r["dialog_id_good"]
    did_b = r["dialog_id_bad"]

    human_g = safe_get(h_lookup, did_g, H_OVERALL)
    human_b = safe_get(h_lookup, did_b, H_OVERALL)
    llm_g   = safe_get(l_lookup, did_g, LLM_OVERALL)
    llm_b   = safe_get(l_lookup, did_b, LLM_OVERALL)

    out = dict(r)
    out["human_delta_overall"] = float(human_g - human_b) if np.isfinite(human_g) and np.isfinite(human_b) else np.nan
    out["llm_good_overall"] = float(llm_g) if np.isfinite(llm_g) else np.nan
    out["llm_bad_overall"]  = float(llm_b) if np.isfinite(llm_b) else np.nan
    out["llm_delta_overall"] = float(llm_g - llm_b) if np.isfinite(llm_g) and np.isfinite(llm_b) else np.nan

    # optional: add subscale deltas + sign consistency counts
    pos_h, pos_l = 0, 0
    n_h, n_l = 0, 0

    for k in subscales_pos:
        hk = f"human_gold_{k}"
        lk = f"llm_median_{k}"
        if hk in hgold.columns:
            hg = safe_get(h_lookup, did_g, hk)
            hb = safe_get(h_lookup, did_b, hk)
            out[f"human_delta_{k}"] = float(hg - hb) if np.isfinite(hg) and np.isfinite(hb) else np.nan
            if np.isfinite(out[f"human_delta_{k}"]):
                n_h += 1
                if out[f"human_delta_{k}"] > 0:
                    pos_h += 1
        if lk in llm.columns:
            lg = safe_get(l_lookup, did_g, lk)
            lb = safe_get(l_lookup, did_b, lk)
            out[f"llm_delta_{k}"] = float(lg - lb) if np.isfinite(lg) and np.isfinite(lb) else np.nan
            if np.isfinite(out[f"llm_delta_{k}"]):
                n_l += 1
                if out[f"llm_delta_{k}"] > 0:
                    pos_l += 1

    out["human_subscale_pos_frac"] = (pos_h / n_h) if n_h else np.nan
    out["llm_subscale_pos_frac"]   = (pos_l / n_l) if n_l else np.nan

    rows.append(out)

pair_table = pd.DataFrame(rows)

print("Pairs with deltas:", len(pair_table))
pair_table[[
    RECIPE_KEY, "dialog_id_good", "dialog_id_bad",
    "human_delta_overall", "llm_delta_overall",
    "human_subscale_pos_frac", "llm_subscale_pos_frac"
]].head()

Pairs with deltas: 18


,recipe_title,dialog_id_good,dialog_id_bad,human_delta_overall,llm_delta_overall,human_subscale_pos_frac,llm_subscale_pos_frac
0,Apple Pie Filling,D040,D077,4.5,5.0,1.0,1.0
1,Au Gratin-Pumpkin Layered Casserole,D034,D064,3.0,5.0,1.0,1.0
2,Bourbon Wieners,D002,D042,5.5,5.0,1.0,1.0
3,Carrot Casserole,D019,D060,4.5,5.0,1.0,1.0
4,Carrot and Sweet Potato Tzimmes,D037,D071,4.5,5.0,1.0,1.0


## Two-Stages Validation

**Strict PASS/FAIL rules (gate)**

Two stages: **(A) Pre-check gate** (Study A-based) and **(B) Mini-pilot gate** (Study B pilot)

1. **Pre-check PASS/FAIL** (Study A data only)

    PASS if ALL apply:

    * human_delta_overall >= 0.7 (good significantly > bad)
    * llm_delta_overall >= 0.7
    * Directional consistency on subscales:
      * human_subscale_pos_frac >= 0.70 (at least 5/7 subscales in the right direction)
      * llm_subscale_pos_frac >= 0.70

    FAIL if one condition is violated
    * Optionally stricter for final top pool: threshold 1.0 instead of 0.7

2. **Mini-pilot PASS/FAIL** (Study B pilot)
    * For each pair:
    * 1 item: ‘How good was the communication?’
    * and Forced-Choice are collected
  
    PASS if:
    * Good > Bad in expected direction and difference ≥ 0.7 Likert points (or p<.05 for small test, focus more on effect+direction)

## Apply PASS/FAIL + rank top pool

In [ ]:
# hard thresholds
HUMAN_DELTA_MIN = 0.7
LLM_DELTA_MIN   = 0.7
SUBSCALE_POS_FRAC_MIN = 0.70

def apply_precheck_gate(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    d["pass_precheck"] = (
        (d["human_delta_overall"].astype(float) >= HUMAN_DELTA_MIN) &
        (d["llm_delta_overall"].astype(float)   >= LLM_DELTA_MIN) &
        (d["human_subscale_pos_frac"].astype(float) >= SUBSCALE_POS_FRAC_MIN) &
        (d["llm_subscale_pos_frac"].astype(float)   >= SUBSCALE_POS_FRAC_MIN)
    )

    # reason string
    reasons = []
    for _, row in d.iterrows():
        r = []
        if not (np.isfinite(row["human_delta_overall"]) and row["human_delta_overall"] >= HUMAN_DELTA_MIN):
            r.append("fail_human_delta_overall")
        if not (np.isfinite(row["llm_delta_overall"]) and row["llm_delta_overall"] >= LLM_DELTA_MIN):
            r.append("fail_llm_delta_overall")
        if not (np.isfinite(row["human_subscale_pos_frac"]) and row["human_subscale_pos_frac"] >= SUBSCALE_POS_FRAC_MIN):
            r.append("fail_human_subscale_direction")
        if not (np.isfinite(row["llm_subscale_pos_frac"]) and row["llm_subscale_pos_frac"] >= SUBSCALE_POS_FRAC_MIN):
            r.append("fail_llm_subscale_direction")
        reasons.append(", ".join(r))
    d["precheck_fail_reasons"] = reasons
    return d

pair_table = apply_precheck_gate(pair_table)

print("PASS precheck:", int(pair_table["pass_precheck"].sum()))
print("FAIL precheck:", int((~pair_table["pass_precheck"]).sum()))

# Rank score for TOP pool: combine human + llm deltas
# sum of standardized deltas
tmp = pair_table.copy()
tmp["human_delta_z"] = (tmp["human_delta_overall"] - tmp["human_delta_overall"].mean()) / tmp["human_delta_overall"].std(ddof=1)
tmp["llm_delta_z"]   = (tmp["llm_delta_overall"]   - tmp["llm_delta_overall"].mean())   / tmp["llm_delta_overall"].std(ddof=1)
tmp["rank_score"] = tmp["human_delta_z"] + tmp["llm_delta_z"]

# Save full pair table
tmp.to_csv(OUT_PAIR_TABLE, index=False, encoding="utf-8")
print("Saved pair table:", OUT_PAIR_TABLE)

# Keep only PASS pool
pass_pool = tmp[tmp["pass_precheck"]].sort_values("rank_score", ascending=False).copy()
pass_pool.to_csv(OUT_PASS_POOL, index=False, encoding="utf-8")
print("Saved PASS pool:", OUT_PASS_POOL)

pass_pool.head(10)[[
    RECIPE_KEY, "dialog_id_good", "dialog_id_bad",
    "human_delta_overall", "llm_delta_overall", "rank_score"
]]

PASS precheck: 17
FAIL precheck: 1
Saved pair table: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/studyB_recipe_pairs_with_deltas.csv
Saved PASS pool: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/studyB_pass_pool_pairs.csv


,recipe_title,dialog_id_good,dialog_id_bad,human_delta_overall,llm_delta_overall,rank_score
10,Orange Beef Kabobs with Grilled Fruit,D021,D063,6.0,6.0,3.904343
11,Red Bliss Potato Salad with Gorgonzola and Wal...,D007,D048,3.5,6.0,1.854148
12,Sweet and Nutty Pork Chops,D025,D070,3.0,6.0,1.444109
2,Bourbon Wieners,D002,D042,5.5,5.0,0.886623
14,Turkey Tom Kha Gai,D024,D069,5.0,5.0,0.476584
8,Mock Caprese Salad,D033,D062,5.0,5.0,0.476584
0,Apple Pie Filling,D040,D077,4.5,5.0,0.066545
4,Carrot and Sweet Potato Tzimmes,D037,D071,4.5,5.0,0.066545
3,Carrot Casserole,D019,D060,4.5,5.0,0.066545
9,Mom's Margarine Cake,D014,D055,4.5,5.0,0.066545


## Export top pool (17)

In [ ]:
TOP_N = 18
top_pool = pass_pool.head(TOP_N).copy()

top_pool.to_csv(OUT_TOP_POOL, index=False, encoding="utf-8")
print("Saved TOP pool:", OUT_TOP_POOL)
print("TOP pool size:", len(top_pool))

top_pool[[RECIPE_KEY, "dialog_id_good", "dialog_id_bad",
          "human_delta_overall", "llm_delta_overall",
          "human_subscale_pos_frac", "llm_subscale_pos_frac",
          "rank_score"]]

Saved TOP pool: /content/drive/MyDrive/Masterarbeit/Dialoge/StudyA/studyA_exports/studyB_top_pool_pairs.csv
TOP pool size: 17


,recipe_title,dialog_id_good,dialog_id_bad,human_delta_overall,llm_delta_overall,human_subscale_pos_frac,llm_subscale_pos_frac,rank_score
10,Orange Beef Kabobs with Grilled Fruit,D021,D063,6.0,6.0,1.0,1.0,3.904343
11,Red Bliss Potato Salad with Gorgonzola and Wal...,D007,D048,3.5,6.0,1.0,1.0,1.854148
12,Sweet and Nutty Pork Chops,D025,D070,3.0,6.0,1.0,1.0,1.444109
2,Bourbon Wieners,D002,D042,5.5,5.0,1.0,1.0,0.886623
14,Turkey Tom Kha Gai,D024,D069,5.0,5.0,1.0,1.0,0.476584
8,Mock Caprese Salad,D033,D062,5.0,5.0,1.0,1.0,0.476584
0,Apple Pie Filling,D040,D077,4.5,5.0,1.0,1.0,0.066545
4,Carrot and Sweet Potato Tzimmes,D037,D071,4.5,5.0,1.0,1.0,0.066545
3,Carrot Casserole,D019,D060,4.5,5.0,1.0,1.0,0.066545
9,Mom's Margarine Cake,D014,D055,4.5,5.0,1.0,1.0,0.066545


In [ ]:
pair_table.loc[~pair_table["pass_precheck"], [
    "recipe_title",
    "human_delta_overall",
    "llm_delta_overall",
    "human_subscale_pos_frac",
    "llm_subscale_pos_frac",
    "precheck_fail_reasons"
]]

,recipe_title,human_delta_overall,llm_delta_overall,human_subscale_pos_frac,llm_subscale_pos_frac,precheck_fail_reasons
15,Very Old Meatloaf Recipe,2.5,5.0,0.666667,1.0,fail_human_subscale_direction
